# Text contracts and KA-0

Run after Chapters 8 and 9. Bilingual examples live in shared content resources. All documents and orders are fictional; this notebook only reads local files. The five documents, nine queries, relevance labels and failure taxonomy are frozen in ka0-v1.

In [1]:
from pathlib import Path
import sys, json, math
candidates = [Path.cwd(), *Path.cwd().parents]
root = next((p for p in candidates if (p / "data/part-i/ngram.json").is_file()
             and (p / "src/config/book.mjs").is_file()), None)
if root is None:
    raise FileNotFoundError("book repository boundary not found")
sys.path.insert(0, str(root / "code/part-ii"))
print("Repository fixtures found")

Repository fixtures found


## Exact input/output contract

Compare against authored expected strings, then check idempotence. Keep each raw string separately; a search key cannot reconstruct lost case, spacing or code-point offsets.

In [2]:
from text_processing import search_key, describe
contract = json.loads((root / "data/part-ii/text-contract.json").read_text(encoding="utf-8"))
for case in contract["cases"]:
    key = search_key(case["raw"])
    assert key == case["expected"], case["id"]
    assert search_key(key) == key
    print(case["id"], repr(case["raw"]), "=>", repr(key))
for case in contract["display_cases"]:
    actual = describe(case["raw"])
    assert all(actual[k] == case[k] for k in ["code_points", "utf8_hex", "code_point_count", "byte_count"])
    print(actual)
try:
    search_key(None)
except TypeError as error:
    print(type(error).__name__, str(error))

english-whitespace '  Beijing\tLODGING\n750  ' => 'beijing lodging 750'
chinese-whitespace '  北京\t住宿\n750 元  ' => '北京 住宿 750 元'
composed 'café' => 'café'
decomposed 'café' => 'café'
casefold-collision 'Straße' => 'strasse'
casefold-counterpart 'STRASSE' => 'strasse'
width 'Ａ-104' => 'ａ-104'
ascii-identifier 'A-104' => 'a-104'
negation-en 'Do NOT cancel A-104.' => 'do not cancel a-104.'
negation-zh '不要取消 A-104。' => '不要取消 a-104。'
punctuation '750, not 600' => '750, not 600'
zero-width 'a\u200bb' => 'a\u200bb'
empty '\t\n ' => ''
{'code_points': ['U+00E9'], 'utf8_hex': 'c3 a9', 'code_point_count': 1, 'byte_count': 2, 'key': 'é'}
{'code_points': ['U+0065', 'U+0301'], 'utf8_hex': '65 cc 81', 'code_point_count': 2, 'byte_count': 3, 'key': 'é'}
{'code_points': ['U+4E2D'], 'utf8_hex': 'e4 b8 ad', 'code_point_count': 1, 'byte_count': 3, 'key': '中'}
{'code_points': ['U+1F642'], 'utf8_hex': 'f0 9f 99 82', 'code_point_count': 1, 'byte_count': 4, 'key': '🙂'}
TypeError raw must be a string


In [3]:
offset_case = next(row for row in contract["loss_counterexamples"] if row["kind"] == "offsets")
raw, key = offset_case["before"], search_key(offset_case["before"])
assert raw.index("750") == 6 and key.index("750") == 5
print("raw span:", raw[6:9], "incorrect reused normalized offsets:", repr(raw[5:8]))

raw span: 750 incorrect reused normalized offsets: ' 75'


## Three baselines, one collection

The expected count matrix and predictions were enumerated separately from the implementation. Print every query score, ranking and top-one prediction. A null prediction is abstention; an all-zero diagnostic ranking is not evidence.

In [4]:
sys.path.insert(0, str(root / "code/knowledge-assistant"))
from baselines import run, load_fixture, run_locale, rank
result = run(write=False)
expected = json.loads((root / "data/knowledge-assistant/ka0-expected.json").read_text())
for locale, rows in result["locales"].items():
    assert rows["count_matrix"] == expected["count_matrix"]
    assert rows["document_frequency"] == expected["document_frequency"]
    print(locale, "count matrix:", rows["count_matrix"])
    print(locale, "TF-IDF matrix:", rows["tfidf_matrix"])
    for method, queries in rows["methods"].items():
        assert [q["prediction"] for q in queries] == expected[method + "_predictions"]
        for query in queries:
            print(locale, method, query)
assert result["locales"]["en"]["success_counts"] == {"keyword": 5, "count": 6, "tfidf": 7}

{
  "en": {
    "success_counts": {
      "keyword": 5,
      "count": 6,
      "tfidf": 7
    },
    "query_count": 9,
    "predictions": {
      "keyword": [
        "travel-v1",
        "travel-v1",
        "travel-v2",
        null,
        "travel-v1",
        "status-faq-v1",
        "approval-faq-v1",
        "travel-v1",
        null
      ],
      "count": [
        "travel-v2",
        "rail-faq-v1",
        "travel-v2",
        null,
        "travel-v1",
        "status-faq-v1",
        "approval-faq-v1",
        "travel-v1",
        null
      ],
      "tfidf": [
        "travel-v2",
        "status-faq-v1",
        "travel-v2",
        null,
        "travel-v1",
        "status-faq-v1",
        "approval-faq-v1",
        "travel-v1",
        null
      ]
    }
  },
  "zh-hans": {
    "success_counts": {
      "keyword": 5,
      "count": 6,
      "tfidf": 7
    },
    "query_count": 9,
    "predictions": {
      "keyword": [
        "travel-v1",
        "travel-v1",
      

## Transfer: vocabulary and corpus are part of the model

Append a new unrelated document to a separate variant. The old Beijing document frequency stays 3, but N changes from 5 to 6, so its IDF increases from 1+ln(6/4) to 1+ln(7/4). Do not overwrite the frozen evaluation fixture.

In [5]:
import copy
variant = copy.deepcopy(load_fixture())
extra = copy.deepcopy(variant["documents"][-1])
extra["id"] = "diagnostic-extra"
extra["index_text"] = {locale: "unrelated" for locale in ["en", "zh-hans"]}
variant["documents"].append(extra)
variant["candidate_order"].append(extra["id"])
changed = run_locale(variant, "en")
assert changed["document_frequency"][0] == 3
assert math.isclose(changed["idf"][0], 1 + math.log(7/4), abs_tol=1e-12)
print("original/variant IDF:", result["locales"]["en"]["idf"][0], changed["idf"][0])
assert rank([0, 0], ["first", "second"])["prediction"] is None
assert rank([1, 1], ["first", "second"])["prediction"] == "first"

original/variant IDF: 1.4054651081081644 1.5596157879354227
